<a href="https://colab.research.google.com/github/Titantus/The-T0C-Predictive-Routing-Engine/blob/main/Project_Nutcracker_Phase_III.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Consolidated Simulation Engine Overview

## Overview
This notebook presents two distinct yet complementary simulation frameworks developed under 'Project Nutcracker, Phase III', each addressing different aspects of Green River kerogen extraction. Phase III marks a strategic shift from the unfeasible terahertz (THz) approach of Phase II towards physically grounded methods. The goal of this consolidation is to improve the structure and clarity of the notebook.

### 1. Cryo-Mechanical Green River Extraction Simulator
**Purpose:** This simulator (formerly the primary focus of 'Project Nutcracker, Phase III') models a simplified, physically grounded method for Green River kerogen extraction. It focuses on the synergistic effects of cryogenic embrittlement, mechanical fracture, and electromigration assist.
**Current State:** It provides quantitative insights into processing efficiency, throughput, and yield enhancement, serving as a 'usable research tool' for experimental validation and optimization of key parameters (e.g., freezing rate, current density).

### 2. Staged Multi-Field Potential and Bond-Activation Simulation Framework
**Purpose:** This framework is designed as a causal pipeline to simulate the intricate effects of various physical fields (electrical, cryogenic, mechanical, electromagnetic) on material response and subsequent chemical outcomes. It aims to provide a mechanistic understanding of bond activation processes.
**Current State:** It is presented as a simulation architecture where individual stages can be configured and studied for ablation analyses. Many parameters within this framework are currently phenomenological and require calibration against experimental data for full validation.

## Rationale for Phase III: Shift to Established Physics
Phase III re-focuses on three established physical effects, eliminating the need for THz hardware:
1.  **Cryogenic embrittlement**: Exploiting shale's brittle behavior at low temperatures.
2.  **Mechanical fracture**: Utilizing stress-concentration fracture with a diamond-toothed rotor.
3.  **Electromigration-based electrochemical assist**: Employing electron-wind forces for atom movement at high current densities.

This consolidated approach simplifies the experimental pathway and provides both high-level predictive power (from the Cryo-Mechanical Simulator) and detailed mechanistic insights (from the Staged Framework), making the research more tractable and robust for future development.

In [ ]:
# @title Project Nutcracker Phase III: Cryo-Mechanical Green River Extraction Simulator
# Synergistic model integrating cryogenic embrittlement, mechanical fracture,
# thermal-mechanical stress, and electromigration assist per TZOC-NUTCRACKER-PHASE3-2026-R1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Literal, Optional
import os
from scipy.optimize import minimize

plt.style.use('dark_background')

# ================================================================
# GLOBAL CONSTANTS (Literature-aligned from Phase III doc)
# ================================================================

LN2_PRODUCTION_ENERGY_KWH_PER_KG: float = 0.9
LN2_LATENT_HEAT_KJ_PER_KG: float = 200.0
SHALE_SPECIFIC_HEAT_KJ_PER_KGK: float = 0.8
MU0: float = 4 * np.pi * 1e-7

# Brittleness from 2025 cryogenic shale study (freezing rate thresholds)
def brittleness_from_freezing_rate(rate_c_per_s: float) -> float:
    """Enhanced brittleness <2.5 °C/s; degraded >5 °C/s."""
    if rate_c_per_s < 0:
        raise ValueError("Freezing rate cannot be negative.")
    if rate_c_per_s < 2.5:
        return 0.18  # Enhanced
    elif rate_c_per_s > 5.0:
        return 0.08  # Degraded
    return 0.12  # Transitional

# ================================================================
# CENTRAL CONFIGURATION
# ================================================================

@dataclass
class NutcrackerConfig:
    """Centralized config for Phase III cryo-mechanical + assist models."""
    # Process flow
    shale_mass_kg_per_hr: float = 120.0
    freezing_rate_c_per_s: float = 2.0          # Key control param (<2.5 ideal)
    cycles: int = 3
    initial_mass_kg: float = 10.0
    delta_temp_c: float = 150.0                 # Thermal shock

    # Mechanical properties (Green River marlstone/kerogen approx)
    alpha_per_c: float = 10e-6
    youngs_modulus_pa: float = 40e9
    poisson_ratio: float = 0.20
    fracture_strength_mpa: float = 50.0

    # Grinding
    feed_rate_kg_per_hr: float = 150.0
    rotor_rpm: float = 4000.0
    rotor_torque_nm: float = 200.0
    max_rotor_power_kw: float = 500.0
    fracture_propagation_efficiency: float = 0.8
    throughput_scaling: float = 1000.0

    # Electrochemical / Electromigration assist
    current_density_a_per_m2: float = 1e6       # Realistic bulk target (scale-up question)
    material_resistivity_ohm_m: float = 1e-4    # After conductive doping (graphite/iron)
    activation_energy_joules: float = 1.6e-19   # ~1 eV
    dopant: Literal["none", "graphite", "iron"] = "none" # New: Type of conductive dopant
    magnetic_recovery_efficiency: float = 0.0   # New: Efficiency for magnetic separation if iron is used (0-1)

    # Type I Kerogen chain model (end-cleavage preference)
    initial_avg_chain_length: float = 50.0
    chain_length_std_dev: float = 15.0
    num_chains: int = 10000
    end_cleavage_probability: float = 0.3
    cryo_enhancement_factor: float = 1.5
    shorter_chain_oxygen_reduction_factor: float = 0.8

    # LN2 & system
    ambient_temp_c: float = 25.0
    ln2_boiling_point_c: float = -196.0
    heat_exchanger_efficiency: float = 0.80
    ln2_boil_off_loss_percent: float = 0.05
    ln2_recovery_rate: float = 0.75

    # EM / Square wave (for potential assist)
    fundamental_freq_hz: float = 50e3
    conductivity_s_per_m: float = 5000.0
    pellet_radius_m: float = 0.01
    max_harmonics: int = 5

    def __post_init__(self):
        # Basic validation (expand as needed)
        for field in ['shale_mass_kg_per_hr', 'freezing_rate_c_per_s', 'delta_temp_c',
                      'youngs_modulus_pa', 'fracture_strength_mpa', 'rotor_rpm']:
            if getattr(self, field) <= 0:
                raise ValueError(f"{field} must be positive.")

        if not (0 <= self.magnetic_recovery_efficiency <= 1):
            raise ValueError("magnetic_recovery_efficiency must be between 0 and 1.")

        # Adjust properties based on dopant
        if self.dopant == "graphite":
            self.material_resistivity_ohm_m = 8e-5  # Lower resistivity
        elif self.dopant == "iron":
            self.material_resistivity_ohm_m = 5e-5  # Even lower resistivity
            self.activation_energy_joules *= 0.95 # Slight reduction in activation energy due to catalytic effects
            self.magnetic_recovery_efficiency = 0.8 # Assume 80% recovery for iron doping


# ================================================================
# PHYSICS MODELS (Synergistic & Document-Aligned)
# ================================================================

def calculate_thermal_shock_stress(cfg: NutcrackerConfig) -> Tuple[float, bool]:
    """Plane-strain thermal stress + fracture check."""
    stress_pa = (cfg.youngs_modulus_pa * cfg.alpha_per_c * cfg.delta_temp_c) / (1.0 - cfg.poisson_ratio)
    stress_mpa = stress_pa / 1e6
    return stress_mpa, stress_mpa >= cfg.fracture_strength_mpa


def simulate_freeze_shattering_cycles(cfg: NutcrackerConfig) -> List[Dict[str, Any]]:
    """Progressive mass reduction via cryo-embrittlement."""
    shatter_rate = brittleness_from_freezing_rate(cfg.freezing_rate_c_per_s)
    current_mass = cfg.initial_mass_kg
    history = []
    for c in range(1, cfg.cycles + 1):
        shed = current_mass * shatter_rate
        retained = current_mass - shed
        history.append({
            "cycle": c,
            "shed_kg": round(shed, 3),
            "retained_kg": round(retained, 3),
            "brittleness_factor": shatter_rate,
        })
        current_mass = retained
    return history


def grinding_score(grit_um: float, rpm: float, pressure_mpa: float) -> float:
    """Fracture promotion vs heat penalty."""
    fracture_term = (grit_um ** 0.35) * (pressure_mpa ** 0.6)
    heat_penalty = (rpm / 9000.0) ** 2
    return fracture_term - heat_penalty


def optimize_grinding_parameters() -> Tuple[Tuple[float, float, float], float]:
    """Grid search for optimal mechanical parameters."""
    grit_sizes = np.linspace(50, 500, 30)
    rpms = np.linspace(1500, 9000, 30)
    pressures = np.linspace(5.0, 35.0, 30)
    best_score = -np.inf
    best_params = (0.0, 0.0, 0.0)
    for g in grit_sizes:
        for r in rpms:
            for p in pressures:
                score = grinding_score(g, r, p)
                if score > best_score:
                    best_score = score
                    best_params = (g, r, p)
    return best_params, best_score


def calculate_ln2_cost(cfg: NutcrackerConfig) -> Dict[str, float]:
    """Energy & LN2 consumption for cryogenic phase."""
    delta_t = cfg.ambient_temp_c - cfg.ln2_boiling_point_c
    energy_cool_kj_hr = cfg.shale_mass_kg_per_hr * SHALE_SPECIFIC_HEAT_KJ_PER_KGK * delta_t
    energy_cool_kwh_hr = energy_cool_kj_hr / 3600.0

    ln2_mass_cooling = energy_cool_kj_hr / LN2_LATENT_HEAT_KJ_PER_KG / cfg.heat_exchanger_efficiency
    ln2_lost = ln2_mass_cooling * cfg.ln2_boil_off_loss_percent * (1 - cfg.ln2_recovery_rate)
    total_ln2 = ln2_mass_cooling + ln2_lost
    total_energy = total_ln2 * LN2_PRODUCTION_ENERGY_KWH_PER_KG

    return {
        "total_energy_kwh_per_hr": total_energy,
        "total_ln2_kg_per_hr": total_ln2,
        "energy_cost_per_kg_shale": total_energy / cfg.shale_mass_kg_per_hr,
        "cooling_energy_kwh_per_hr": energy_cool_kwh_hr,
    }


def calculate_mechanical_throughput(cfg: NutcrackerConfig) -> float:
    """Power-limited throughput, boosted by brittleness."""
    rotor_power_kw = (cfg.rotor_torque_nm * cfg.rotor_rpm) / 9549.3
    effective_kw = min(rotor_power_kw, cfg.max_rotor_power_kw)

    throughput = (
        effective_kw
        * 0.7  # GRINDING_EFFICIENCY
        * brittleness_from_freezing_rate(cfg.freezing_rate_c_per_s)
        * cfg.fracture_propagation_efficiency
        * 0.5  # MATERIAL_STRENGTH_FACTOR
        * cfg.throughput_scaling
    )
    return min(throughput, cfg.feed_rate_kg_per_hr)


def simulate_chain_liberation(cfg: NutcrackerConfig) -> Dict[str, Any]:
    """Stochastic Type I kerogen end-cleavage model."""
    rng = np.random.default_rng(42)
    initial = rng.normal(cfg.initial_avg_chain_length, cfg.chain_length_std_dev, cfg.num_chains)
    initial = np.maximum(1, initial).astype(int)

    cleavage_mask = rng.random(cfg.num_chains) < (cfg.end_cleavage_probability * cfg.cryo_enhancement_factor)
    liberated = initial.copy()
    liberated[cleavage_mask] = np.maximum(
        1, liberated[cleavage_mask] * rng.uniform(0.3, 0.7, cleavage_mask.sum())
    ).astype(int)

    initial_o = np.full(cfg.num_chains, 10.0)
    liberated_o = initial_o.copy()
    liberated_o[cleavage_mask] *= cfg.shorter_chain_oxygen_reduction_factor

    return {
        "initial_chains": initial,
        "liberated_chains": liberated,
        "cleavage_mask": cleavage_mask,
        "initial_oxygen_content": initial_o,
        "liberated_oxygen_content": liberated_o,
        "liberated_fraction": cleavage_mask.mean(),
    }


def calculate_electromigration_effects(cfg: NutcrackerConfig) -> Dict[str, float]:
    """Electron-wind force & yield boost (document-scoped)."""
    electron_wind = cfg.current_density_a_per_m2 * cfg.material_resistivity_ohm_m * 1e-10
    potential_red = electron_wind * 5e4
    max_red = cfg.activation_energy_joules * 0.9
    reduction = min(potential_red, max_red)
    reduced_ea = max(cfg.activation_energy_joules - reduction, cfg.activation_energy_joules * 0.1)
    enhancement = min(cfg.activation_energy_joules / reduced_ea, 1000.0)

    return {
        "electron_wind_force": electron_wind,
        "activation_reduction": reduction,
        "reduced_ea": reduced_ea,
        "yield_enhancement": enhancement,
    }


# ================================================================
# MAIN SIMULATION
# ================================================================

def run_nutcracker_simulation(cfg: NutcrackerConfig) -> Dict[str, Any]:
    """Orchestrates synergistic cryo-mechanical + assist simulation."""
    results = {"config": cfg}

    results["thermal_shock"] = calculate_thermal_shock_stress(cfg)
    results["freeze_shatter"] = simulate_freeze_shattering_cycles(cfg)
    results["grinding_opt"] = optimize_grinding_parameters()
    results["ln2"] = calculate_ln2_cost(cfg)
    results["mechanical_throughput"] = calculate_mechanical_throughput(cfg)
    results["chain_liberation"] = simulate_chain_liberation(cfg)
    results["electromigration"] = calculate_electromigration_effects(cfg)

    # Add magnetic recovery potential
    if cfg.dopant == "iron":
        # Assuming that the magnetic recovery acts on the mass processed
        results["magnetic_recovery_potential_kg_hr"] = results["mechanical_throughput"] * cfg.magnetic_recovery_efficiency
    else:
        results["magnetic_recovery_potential_kg_hr"] = 0.0

    # Overall synergy score (higher = better cryo-mechanical-electro coupling)
    synergy = (
        brittleness_from_freezing_rate(cfg.freezing_rate_c_per_s) *
        results["chain_liberation"]["liberated_fraction"] *
        results["electromigration"]["yield_enhancement"] ** 0.3  # sub-linear
    )
    results["overall_synergy_score"] = synergy

    return results


def example_sweep() -> pd.DataFrame:
    """Sweep freezing rate — the key actionable parameter."""
    rates = np.linspace(1.0, 6.0, 8)
    data = []
    for rate in rates:
        # Ensure we use the current config's dopant for the sweep if it's set in the main execution block
        # or default to 'none' if this function is called independently
        current_dopant = NutcrackerConfig().dopant # Default to none if not explicitly passed
        # This needs to be more robust if example_sweep is intended to run with the global cfg
        # For now, let's assume it runs with a default 'none' dopant if not explicitly configured
        cfg = NutcrackerConfig(freezing_rate_c_per_s=rate, shale_mass_kg_per_hr=100.0, dopant='none') # Explicitly set dopant for sweep consistency
        res = run_nutcracker_simulation(cfg)
        data.append({
            "freezing_rate_c_per_s": rate,
            "throughput_kg_hr": res["mechanical_throughput"],
            "liberated_fraction": res["chain_liberation"]["liberated_fraction"],
            "synergy_score": res["overall_synergy_score"],
            "ln2_cost_kwh_per_kg": res["ln2"]["energy_cost_per_kg_shale"],
        })
    return pd.DataFrame(data)


def plot_results(results: Dict[str, Any], sweep_df: Optional[pd.DataFrame] = None, title_suffix: str = ""):
    """Visual dashboard aligned with Phase III priorities."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Project Nutcracker Phase III Simulation{title_suffix}', fontsize=16)

    # Freeze-shatter
    fs_df = pd.DataFrame(results['freeze_shatter'])
    axes[0, 0].plot(fs_df['cycle'], fs_df['retained_kg'], 'o-', color='cyan', label='Retained')
    axes[0, 0].set_title('Cryo-Embrittlement Mass Reduction')
    axes[0, 0].set_xlabel('Cycle'); axes[0, 0].set_ylabel('Mass (kg)')
    axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.5)

    # Chain liberation
    cl = results['chain_liberation']
    axes[0, 1].hist(cl['initial_chains'], bins=20, alpha=0.7, label='Initial', color='gold')
    axes[0, 1].hist(cl['liberated_chains'][cl['cleavage_mask']], bins=20, alpha=0.7, label='Liberated', color='magenta')
    axes[0, 1].set_title('Kerogen Chain Lengths (End-Cleavage)')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.5)

    # Sweep / Placeholder for optimized results
    if sweep_df is not None and not sweep_df.empty:
        axes[1, 0].plot(sweep_df['freezing_rate_c_per_s'], sweep_df['throughput_kg_hr'], 'x--', color='lime', label='Throughput')
        axes[1, 0].plot(sweep_df['freezing_rate_c_per_s'], sweep_df['synergy_score'], 's-', color='orange', label='Synergy')
        axes[1, 0].set_title('Freezing Rate Impact (Key Lever)')
        axes[1, 0].set_xlabel('Freezing Rate (°C/s)'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.5)
    else:
        # Display optimized freezing rate and current density if no sweep data
        optimized_fr = results['config'].freezing_rate_c_per_s
        optimized_cd = results['config'].current_density_a_per_m2
        axes[1, 0].text(0.5, 0.7, f'Optimized Freezing Rate: {optimized_fr:.2f} °C/s',
                       horizontalalignment='center', verticalalignment='center', transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].text(0.5, 0.5, f'Optimized Current Density: {optimized_cd:.1e} A/m^2',
                       horizontalalignment='center', verticalalignment='center', transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].set_title('Optimized Parameters')
        axes[1, 0].axis('off') # Hide axes for cleaner text display

    # Electromigration
    em = results['electromigration']
    axes[1, 1].bar(['Yield Enhancement'], [em['yield_enhancement']], color='violet')
    axes[1, 1].set_title('Electromigration Assist')
    axes[1, 1].grid(True, alpha=0.5)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


def export_results(results: Dict[str, Any], sweep_df: pd.DataFrame, output_dir: str = 'nutcracker_exports'):
    os.makedirs(output_dir, exist_ok=True)
    pd.DataFrame([results['config'].__dict__]).to_csv(f'{output_dir}/config.csv', index=False)
    pd.DataFrame(results['freeze_shatter']).to_csv(f'{output_dir}/freeze_shatter.csv', index=False)
    sweep_df.to_csv(f'{output_dir}/freezing_rate_sweep.csv', index=False)

    # Export additional results like magnetic recovery if present
    if 'magnetic_recovery_potential_kg_hr' in results:
        pd.DataFrame([{'magnetic_recovery_potential_kg_hr': results['magnetic_recovery_potential_kg_hr']}]).to_csv(f'{output_dir}/magnetic_recovery_potential.csv', index=False)

    print(f"✅ Results exported to {output_dir}/")


def optimize_scenario_multi_objective(base_cfg: NutcrackerConfig) -> Dict[str, Any]:
    """Multi-objective optimization for liberated fraction vs. LN2 cost."""

    # Objective function to maximize (liberated fraction / LN2 cost)
    # We minimize the negative of this ratio
    def objective(params):
        freezing_rate, current_density = params
        # Create a new config instance for each optimization step to avoid modifying the base_cfg repeatedly
        cfg = NutcrackerConfig(**base_cfg.__dict__)
        cfg.freezing_rate_c_per_s = freezing_rate
        cfg.current_density_a_per_m2 = current_density

        res = run_nutcracker_simulation(cfg)
        liberated_fraction = res['chain_liberation']['liberated_fraction']
        ln2_cost_kwh_per_kg = res['ln2']['energy_cost_per_kg_shale']

        # Add a small epsilon to avoid division by zero if cost is ever 0
        # and ensure cost is positive. Also, handle cases where liberated_fraction might be 0
        if ln2_cost_kwh_per_kg <= 0 or liberated_fraction <= 0:
            return float('inf') # Penalize invalid or non-productive scenarios

        # Maximize (liberated_fraction / ln2_cost_kwh_per_kg)
        return - (liberated_fraction / ln2_cost_kwh_per_kg)

    # Initial guess for optimization parameters (freezing_rate, current_density)
    initial_guess = [base_cfg.freezing_rate_c_per_s, base_cfg.current_density_a_per_m2]

    # Bounds for parameters (min, max)
    # Freezing rate: 1.0 to 5.0 C/s (to stay within enhanced/transitional brittleness range)
    # Current density: 1e5 to 5e6 A/m^2 (realistic range)
    bounds = [(1.0, 5.0), (1e5, 5e6)]

    # Perform optimization
    result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')

    # Extract optimized parameters
    optimized_freezing_rate = result.x[0]
    optimized_current_density = result.x[1]

    # Run simulation with optimized parameters
    # Create a new config for the final optimized run to ensure clean state
    optimized_cfg = NutcrackerConfig(**base_cfg.__dict__)
    optimized_cfg.freezing_rate_c_per_s = optimized_freezing_rate
    optimized_cfg.current_density_a_per_m2 = optimized_current_density
    optimized_results = run_nutcracker_simulation(optimized_cfg)

    optimized_results['optimization_status'] = result.message
    optimized_results['optimized_value'] = -result.fun # Convert back to positive for interpretation

    return optimized_results

# ================================================================
# EXECUTION
# ================================================================

if __name__ == "__main__":
    base_cfg = NutcrackerConfig(
        freezing_rate_c_per_s=2.2,   # Sweet spot per literature
        rotor_rpm=4200.0,
        current_density_a_per_m2=5e5,
        dopant="iron" # Testing with iron dopant
    )

    # Run initial simulation with base config
    results = run_nutcracker_simulation(base_cfg)
    sweep_df = example_sweep()

    print("=== Project Nutcracker Phase III Results (Base Config) ===")
    print(f"Thermal Shock Stress: {results['thermal_shock'][0]:.1f} MPa | Fracture: {results['thermal_shock'][1]}")
    print(f"Mechanical Throughput: {results['mechanical_throughput']:.1f} kg/hr")
    print(f"LN2 Cost: {results['ln2']['energy_cost_per_kg_shale']:.3f} kWh/kg")
    print(f"Kerogen Liberated Fraction: {results['chain_liberation']['liberated_fraction']:.1%}")
    print(f"Electromigration Yield Boost: {results['electromigration']['yield_enhancement']:.1f}x")
    print(f"Overall Synergy Score: {results['overall_synergy_score']:.3f}")
    if base_cfg.dopant == "iron":
        print(f"Magnetic Recovery Potential: {results['magnetic_recovery_potential_kg_hr']:.1f} kg/hr (with {base_cfg.magnetic_recovery_efficiency*100:.0f}% efficiency)")

    plot_results(results, sweep_df, title_suffix=" (Base Config)")
    export_results(results, sweep_df, output_dir='nutcracker_exports_base')

    print("\n=== Running Multi-Objective Optimization ===")
    optimized_results = optimize_scenario_multi_objective(base_cfg)

    print("\n=== Optimized Results ===")
    print(f"Optimization Status: {optimized_results['optimization_status']}")
    print(f"Optimized Freezing Rate: {optimized_results['config'].freezing_rate_c_per_s:.2f} °C/s")
    print(f"Optimized Current Density: {optimized_results['config'].current_density_a_per_m2:.1e} A/m^2")
    print(f"Optimized Ratio (Liberated Fraction / LN2 Cost): {optimized_results['optimized_value']:.3f}")
    print(f"\nThermal Shock Stress: {optimized_results['thermal_shock'][0]:.1f} MPa | Fracture: {optimized_results['thermal_shock'][1]}")
    print(f"Mechanical Throughput: {optimized_results['mechanical_throughput']:.1f} kg/hr")
    print(f"LN2 Cost: {optimized_results['ln2']['energy_cost_per_kg_shale']:.3f} kWh/kg")
    print(f"Kerogen Liberated Fraction: {optimized_results['chain_liberation']['liberated_fraction']:.1%}")
    print(f"Electromigration Yield Boost: {optimized_results['electromigration']['yield_enhancement']:.1f}x")
    print(f"Overall Synergy Score: {optimized_results['overall_synergy_score']:.3f}")
    if optimized_results['config'].dopant == "iron":
        print(f"Magnetic Recovery Potential: {optimized_results['magnetic_recovery_potential_kg_hr']:.1f} kg/hr (with {optimized_results['config'].magnetic_recovery_efficiency*100:.0f}% efficiency)")

    # Plot and export optimized results
    plot_results(optimized_results, None, title_suffix=" (Optimized Config)") # Pass None for sweep_df for optimized plot
    export_results(optimized_results, sweep_df, output_dir='nutcracker_exports_optimized')


In [ ]:
# @title Project Nutcracker Phase III — Staged Causal Pipeline Simulator
# Causal: ΔV → Current/Power → Energy Deposition → Material Response → Chemical Outcome

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
import os

plt.style.use('dark_background')

# ================================================================
# CONSTANTS
# ================================================================
MU0 = 4 * np.pi * 1e-7
EV_TO_J = 1.602176634e-19


# ================================================================
# CONFIGS
# ================================================================

@dataclass
class MaterialConfig:
    initial_mass_kg: float = 10.0
    conductivity_s_per_m: float = 5000.0
    resistivity_ohm_m: float = 1e-4
    youngs_modulus_pa: float = 40e9
    poisson_ratio: float = 0.20
    fracture_strength_mpa: float = 50.0
    thermal_expansion_per_c: float = 10e-6
    initial_avg_chain_length: float = 50.0
    chain_length_std_dev: float = 15.0
    num_chains: int = 10000


@dataclass
class InterfaceConfig:
    water_wick_height_m: float = 0.3
    oil_wick_height_m: float = 0.3
    water_conductivity_s_per_m: float = 4.8
    permittivity_water: float = 80.1
    permittivity_oil: float = 2.1
    interfacial_tension_mN_m: float = 30.0
    zeta_water_mv: float = 25.0
    zeta_oil_mv: float = 2.0


@dataclass
class ElectricalConfig:
    external_voltage_v: float = 0.0
    load_resistance_ohm: float = 1000.0
    active_area_m2: float = 1e-4


@dataclass
class CryoConfig:
    freezing_rate_c_per_s: float = 2.0
    ambient_temp_c: float = 25.0
    target_temp_c: float = -196.0
    cycles: int = 3


@dataclass
class MechanicalConfig:
    rotor_torque_nm: float = 200.0
    rotor_rpm: float = 4000.0
    max_rotor_power_kw: float = 500.0
    grinding_efficiency: float = 0.7
    fracture_propagation_efficiency: float = 0.8


@dataclass
class EMConfig:
    fundamental_freq_hz: float = 50e3
    max_harmonics: int = 5
    magnetic_field_tesla: float = 0.0  # Optional


@dataclass
class ChemicalConfig:
    base_fragmentation_prob: float = 0.30
    activation_energy_ev: float = 2.5
    energy_coupling_efficiency: float = 0.01  # Phenomenological


@dataclass
class ExperimentConfig:
    """Toggle individual mechanisms for ablation studies."""
    enable_wicking_interface: bool = True
    enable_external_voltage: bool = False
    enable_cryo: bool = True
    enable_mechanical: bool = True
    enable_em: bool = False


# ================================================================
# STAGE FUNCTIONS
# ================================================================

def stage_passive_potential(interface: InterfaceConfig, exp: ExperimentConfig) -> Dict[str, float]:
    if not exp.enable_wicking_interface:
        return {"passive_delta_v_mv": 0.0, "water_mv": 0.0, "oil_mv": 0.0, "interface_mv": 0.0}

    water_mv = 25.0 * interface.water_wick_height_m / (1 + interface.water_conductivity_s_per_m * 1e-4)
    oil_mv = 2.0 * interface.oil_wick_height_m / (1 + interface.water_conductivity_s_per_m * 1e-4)
    interface_mv = interface.interfacial_tension_mN_m / (interface.permittivity_water / interface.permittivity_oil)

    passive_delta_v_mv = water_mv - oil_mv + interface_mv
    return {"passive_delta_v_mv": passive_delta_v_mv, "water_mv": water_mv, "oil_mv": oil_mv, "interface_mv": interface_mv}


def stage_electrical_transport(passive: Dict[str, float], elec: ElectricalConfig, exp: ExperimentConfig) -> Dict[str, float]:
    passive_v = passive["passive_delta_v_mv"] / 1000.0
    total_v = passive_v + (elec.external_voltage_v if exp.enable_external_voltage else 0.0)

    current_a = total_v / elec.load_resistance_ohm
    power_w = total_v * current_a
    current_density = current_a / elec.active_area_m2

    return {
        "total_delta_v_v": total_v,
        "current_a": current_a,
        "current_density_a_m2": current_density,
        "power_w": power_w,
    }


def brittleness_from_rate(rate: float) -> float:
    if rate < 2.5: return 0.18
    if rate > 5.0: return 0.08
    return 0.12


def stage_cryo(material: MaterialConfig, cryo: CryoConfig, exp: ExperimentConfig) -> Dict[str, float]:
    if not exp.enable_cryo:
        return {"brittleness": 0.05, "thermal_stress_mpa": 0.0, "fracture": False}

    delta_t = cryo.ambient_temp_c - cryo.target_temp_c
    stress_pa = material.youngs_modulus_pa * material.thermal_expansion_per_c * delta_t / (1 - material.poisson_ratio)
    stress_mpa = stress_pa / 1e6
    brittleness = brittleness_from_rate(cryo.freezing_rate_c_per_s)
    fracture = stress_mpa >= material.fracture_strength_mpa

    return {"brittleness": brittleness, "thermal_stress_mpa": stress_mpa, "fracture": fracture}


def stage_mechanical(cryo: Dict[str, float], mech: MechanicalConfig, exp: ExperimentConfig) -> Dict[str, float]:
    if not exp.enable_mechanical:
        return {"surface_multiplier": 1.0, "fracture_activation": 0.0}

    rotor_kw = (mech.rotor_torque_nm * mech.rotor_rpm) / 9549.3
    effective_kw = min(rotor_kw, mech.max_rotor_power_kw)
    activation = cryo["brittleness"] * mech.fracture_propagation_efficiency
    surface_mult = 1.0 + activation * 5.0  # Significant surface increase from fracture

    return {"surface_multiplier": surface_mult, "fracture_activation": activation, "effective_power_kw": effective_kw}


def stage_em(material: MaterialConfig, mech: Dict[str, float], em: EMConfig, exp: ExperimentConfig) -> Dict[str, Any]:
    if not exp.enable_em:
        return {"effective_surface": mech.get("surface_multiplier", 1.0), "harmonics": []}

    harmonics = []
    for n in range(1, em.max_harmonics * 2, 2):
        f = em.fundamental_freq_hz * n
        omega = 2 * np.pi * f
        skin = np.sqrt(2 / (omega * MU0 * material.conductivity_s_per_m))
        harmonics.append({"harmonic": n, "freq_hz": f, "skin_m": skin})

    return {"effective_surface": mech.get("surface_multiplier", 1.0), "harmonics": harmonics}


def stage_energy(elec: Dict[str, float], mech: Dict[str, float], cryo: Dict[str, float], em: Dict[str, Any]) -> Dict[str, float]:
    power = elec["power_w"]
    surface = mech.get("surface_multiplier", 1.0)
    cryo_factor = 1.0 + cryo.get("brittleness", 0.0)
    activation_index = power * surface * cryo_factor * (1.0 + mech.get("fracture_activation", 0.0))
    return {"effective_activation_index": activation_index, "power_w": power, "surface": surface}


def stage_chemical(material: MaterialConfig, energy: Dict[str, float], chem: ChemicalConfig) -> Dict[str, Any]:
    rng = np.random.default_rng(42)
    initial = np.maximum(1, rng.normal(material.initial_avg_chain_length, material.chain_length_std_dev, material.num_chains)).astype(int)

    # Mechanical fragmentation
    frag_prob = min(1.0, chem.base_fragmentation_prob * (energy["surface"] / 3.0))
    frag_mask = rng.random(material.num_chains) < frag_prob
    liberated = initial.copy()
    liberated[frag_mask] = np.maximum(1, liberated[frag_mask] * rng.uniform(0.3, 0.7, frag_mask.sum())).astype(int)

    # Energy-driven oxygen cleavage hypothesis
    bond_j = chem.activation_energy_ev * EV_TO_J
    coupled = energy["effective_activation_index"] * chem.energy_coupling_efficiency
    activation_ratio = coupled / max(bond_j, 1e-30)
    ox_prob = min(1.0, activation_ratio)
    ox_mask = rng.random(material.num_chains) < ox_prob

    return {
        "initial_chains": initial,
        "liberated_chains": liberated,
        "fragmentation_fraction": frag_mask.mean(),
        "oxygen_cleavage_fraction": ox_mask.mean(),
        "activation_ratio": activation_ratio,
        "ox_prob": ox_prob,
    }


# ================================================================
# MASTER RUNNER + ABLATION
# ================================================================

def run_staged_nutcracker(exp: ExperimentConfig) -> Dict[str, Any]:
    material = MaterialConfig()
    interface = InterfaceConfig()
    elec = ElectricalConfig(external_voltage_v=0.5 if exp.enable_external_voltage else 0.0)
    cryo = CryoConfig(freezing_rate_c_per_s=2.2)
    mech = MechanicalConfig()
    em = EMConfig()
    chem = ChemicalConfig()

    results = {}
    results["passive"] = stage_passive_potential(interface, exp)
    results["electrical"] = stage_electrical_transport(results["passive"], elec, exp)
    results["cryo"] = stage_cryo(material, cryo, exp)
    results["mechanical"] = stage_mechanical(results["cryo"], mech, exp)
    results["em"] = stage_em(material, results["mechanical"], em, exp)
    results["energy"] = stage_energy(results["electrical"], results["mechanical"], results["cryo"], results["em"])
    results["chemical"] = stage_chemical(material, results["energy"], chem)

    return results


# Example ablation study
if __name__ == "__main__":
    experiments = {
        "baseline": ExperimentConfig(enable_wicking_interface=False, enable_external_voltage=False, enable_cryo=False, enable_mechanical=False, enable_em=False),
        "cryo_mechanical": ExperimentConfig(enable_wicking_interface=False, enable_external_voltage=False, enable_cryo=True, enable_mechanical=True, enable_em=False),
        "full_electro_cryo_mech": ExperimentConfig(enable_wicking_interface=True, enable_external_voltage=True, enable_cryo=True, enable_mechanical=True, enable_em=True),
    }

    for name, exp_cfg in experiments.items():
        print(f"\n=== {name.upper()} ===")
        res = run_staged_nutcracker(exp_cfg)
        print(f"Passive ΔV: {res['passive']['passive_delta_v_mv']:.2f} mV")
        print(f"Power: {res['electrical']['power_w']:.2e} W")
        print(f"Thermal Stress: {res['cryo']['thermal_stress_mpa']:.1f} MPa")
        print(f"Fragmentation: {res['chemical']['fragmentation_fraction']:.1%}")
        print(f"Predicted O-Cleavage: {res['chemical']['oxygen_cleavage_fraction']:.2%}")
        print(f"Activation Ratio: {res['chemical']['activation_ratio']:.2e}")

In [ ]:
# @title
# Staged Multi-Field Potential and Bond-Activation Simulation Framework

# ================================================================
# STAGED MULTI-FIELD POTENTIAL / MATERIAL RESPONSE SIMULATION
# ================================================================
#
# Causal pipeline:
#
#   Stage 0: Baseline Material Characterization
#       ↓
#   Stage 1: Geometry / Interface Configuration
#       ↓
#   Stage 2: Passive Potential Generation (ΔV)
#       ↓
#   Stage 3: Electrical Transport Under Load
#       ↓
#   Stage 4: Cryogenic Conditioning
#       ↓
#   Stage 5: Mechanical Fracture / Surface Activation
#       ↓
#   Stage 6: Electromagnetic Excitation
#       ↓
#   Stage 7: Energy Transfer
#       ↓
#   Stage 8: Molecular / Chemical Response
#       ↓
#   Stage 9: Experimental Validation Metrics
#
# IMPORTANT:
# This is a simulation architecture. Several parameters remain
# phenomenological and must be calibrated against experimental data.
#
# ================================================================

from dataclasses import dataclass, field, asdict
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


# ================================================================
# GLOBAL CONSTANTS
# ================================================================

MU0 = 4 * np.pi * 1e-7       # H/m
KB = 1.380649e-23            # J/K
EV_TO_J = 1.602176634e-19    # J/eV


# ================================================================
# STAGE 0 — BASELINE MATERIAL CONFIGURATION
# ================================================================

@dataclass
class MaterialConfig:
    """
    Defines the initial material state before any conditioning.
    """

    initial_mass_kg: float = 10.0

    # Electrical properties
    conductivity_s_per_m: float = 5000.0
    resistivity_ohm_m: float = 1e-4

    # Mechanical properties
    youngs_modulus_pa: float = 40e9
    poisson_ratio: float = 0.20
    fracture_strength_mpa: float = 50.0
    thermal_expansion_per_c: float = 10e-6

    # Molecular model
    initial_avg_chain_length: float = 50.0
    chain_length_std_dev: float = 15.0
    num_chains: int = 10_000

    # Nominal bond-energy scale.
    # This is a model parameter and should be replaced with
    # experimentally justified values for the specific bond.
    target_bond_energy_ev: float = 2.5


# ================================================================
# STAGE 1 — GEOMETRY / INTERFACE CONFIGURATION
# ================================================================

@dataclass
class InterfaceConfig:
    """
    Defines the geometry responsible for generating passive
    electrokinetic and interfacial potentials.
    """

    water_wick_height_m: float = 0.30
    oil_wick_height_m: float = 0.30

    water_conductivity_s_per_m: float = 4.8

    permittivity_water: float = 80.1
    permittivity_oil: float = 2.1

    interfacial_tension_mN_m: float = 30.0

    # Simplified effective zeta potentials
    zeta_water_mv: float = 25.0
    zeta_oil_mv: float = 2.0


# ================================================================
# STAGE 2 — PASSIVE POTENTIAL GENERATION
# ================================================================

def calculate_wicking_potential(
    height_m: float,
    conductivity_s_per_m: float,
    zeta_potential_mv: float
) -> float:
    """
    Estimates a simplified streaming/electrokinetic potential.

    NOTE:
    This is an effective model, not a first-principles
    electrokinetic derivation.
    """

    return (
        zeta_potential_mv
        * height_m
        / (1.0 + conductivity_s_per_m * 1e-4)
    )


def calculate_interface_potential(
    cfg: InterfaceConfig
) -> float:
    """
    Estimates an effective oil-water interfacial potential.

    The result is a phenomenological metric that should be
    calibrated experimentally.
    """

    dielectric_ratio = (
        cfg.permittivity_water /
        cfg.permittivity_oil
    )

    return (
        cfg.interfacial_tension_mN_m /
        dielectric_ratio
    )


def stage_2_generate_passive_potential(
    cfg: InterfaceConfig
) -> Dict[str, float]:
    """
    Stage 2 output becomes the electrical input to Stage 3.
    """

    water_potential_mv = calculate_wicking_potential(
        cfg.water_wick_height_m,
        cfg.water_conductivity_s_per_m,
        cfg.zeta_water_mv
    )

    oil_potential_mv = calculate_wicking_potential(
        cfg.oil_wick_height_m,
        cfg.water_conductivity_s_per_m,
        cfg.zeta_oil_mv
    )

    interface_potential_mv = calculate_interface_potential(cfg)

    passive_delta_v_mv = (
        water_potential_mv
        - oil_potential_mv
        + interface_potential_mv
    )

    return {
        "water_potential_mv": water_potential_mv,
        "oil_potential_mv": oil_potential_mv,
        "interface_potential_mv": interface_potential_mv,
        "passive_delta_v_mv": passive_delta_v_mv,
    }


# ================================================================
# STAGE 3 — ELECTRICAL TRANSPORT
# ================================================================

@dataclass
class ElectricalConfig:
    """
    Electrical loading conditions.
    """

    external_voltage_v: float = 0.0
    load_resistance_ohm: float = 1000.0

    current_density_a_per_m2: float = 1e7

    active_area_m2: float = 1e-4


def stage_3_electrical_transport(
    passive_stage: Dict[str, float],
    cfg: ElectricalConfig
) -> Dict[str, float]:
    """
    Converts passive ΔV into an electrical state.

    The external voltage is kept separate from passive ΔV so
    their contributions can be measured independently.
    """

    passive_delta_v_v = (
        passive_stage["passive_delta_v_mv"] / 1000.0
    )

    total_delta_v_v = (
        passive_delta_v_v
        + cfg.external_voltage_v
    )

    current_a = (
        total_delta_v_v /
        cfg.load_resistance_ohm
    )

    power_w = (
        total_delta_v_v *
        current_a
    )

    current_density = (
        current_a /
        cfg.active_area_m2
    )

    return {
        "passive_delta_v_v": passive_delta_v_v,
        "external_voltage_v": cfg.external_voltage_v,
        "total_delta_v_v": total_delta_v_v,
        "current_a": current_a,
        "current_density_a_per_m2": current_density,
        "electrical_power_w": power_w,
    }


# ================================================================
# STAGE 4 — CRYOGENIC CONDITIONING
# ================================================================

@dataclass
class CryogenicConfig:
    ambient_temp_c: float = 25.0
    target_temp_c: float = -196.0

    freezing_rate_c_per_s: float = 2.0

    cycles: int = 3

    cryo_enhancement_factor: float = 1.5


def brittleness_from_freezing_rate(
    rate_c_per_s: float
) -> float:

    if rate_c_per_s < 0:
        raise ValueError(
            "Freezing rate cannot be negative."
        )

    if rate_c_per_s < 2.5:
        return 0.18

    if rate_c_per_s > 5.0:
        return 0.08

    return 0.12


def stage_4_cryo_conditioning(
    material: MaterialConfig,
    cfg: CryogenicConfig
) -> Dict[str, float]:

    delta_temp_c = (
        cfg.ambient_temp_c -
        cfg.target_temp_c
    )

    brittleness = brittleness_from_freezing_rate(
        cfg.freezing_rate_c_per_s
    )

    thermal_stress_pa = (
        material.youngs_modulus_pa
        * material.thermal_expansion_per_c
        * delta_temp_c
        / (1.0 - material.poisson_ratio)
    )

    thermal_stress_mpa = (
        thermal_stress_pa / 1e6
    )

    fracture_threshold_reached = (
        thermal_stress_mpa
        >= material.fracture_strength_mpa
    )

    return {
        "temperature_change_c": delta_temp_c,
        "brittleness_factor": brittleness,
        "thermal_stress_mpa": thermal_stress_mpa,
        "fracture_threshold_reached":
            float(fracture_threshold_reached),
    }


# ================================================================
# STAGE 5 — MECHANICAL ACTIVATION
# ================================================================

@dataclass
class MechanicalConfig:

    rotor_torque_nm: float = 200.0
    rotor_rpm: float = 3000.0
    max_rotor_power_kw: float = 500.0

    grinding_efficiency: float = 0.70
    fracture_propagation_efficiency: float = 0.80

    surface_area_multiplier: float = 1.0


def stage_5_mechanical_activation(
    cryo_stage: Dict[str, float],
    cfg: MechanicalConfig
) -> Dict[str, float]:

    rotor_power_kw = (
        cfg.rotor_torque_nm *
        cfg.rotor_rpm
    ) / 9549.3

    effective_power_kw = min(
        rotor_power_kw,
        cfg.max_rotor_power_kw
    )

    brittleness = (
        cryo_stage["brittleness_factor"]
    )

    fracture_activation = (
        brittleness
        * cfg.fracture_propagation_efficiency
    )

    effective_surface_area_multiplier = (
        cfg.surface_area_multiplier
        * (1.0 + fracture_activation)
    )

    return {
        "rotor_power_kw": rotor_power_kw,
        "effective_power_kw": effective_power_kw,
        "fracture_activation": fracture_activation,
        "surface_area_multiplier":
            effective_surface_area_multiplier,
    }


# ================================================================
# STAGE 6 — ELECTROMAGNETIC EXCITATION
# ================================================================

@dataclass
class EMConfig:

    fundamental_freq_hz: float = 50e3
    max_harmonics: int = 5

    magnetic_field_tesla: float = 5.0


def calculate_skin_depth(
    frequency_hz: float,
    conductivity_s_per_m: float
) -> float:

    omega = 2 * np.pi * frequency_hz

    return np.sqrt(
        2.0 /
        (
            omega
            * MU0
            * conductivity_s_per_m
        )
    )


def stage_6_em_excitation(
    material: MaterialConfig,
    mechanical_stage: Dict[str, float],
    cfg: EMConfig
) -> Dict[str, Any]:

    harmonics = []

    for n in range(
        1,
        cfg.max_harmonics * 2,
        2
    ):

        frequency = (
            cfg.fundamental_freq_hz *
            n
        )

        skin_depth = calculate_skin_depth(
            frequency,
            material.conductivity_s_per_m
        )

        harmonics.append({
            "harmonic": n,
            "frequency_hz": frequency,
            "skin_depth_m": skin_depth,
        })

    effective_surface_area = (
        mechanical_stage[
            "surface_area_multiplier"
        ]
    )

    return {
        "harmonics": harmonics,
        "magnetic_field_tesla":
            cfg.magnetic_field_tesla,
        "effective_surface_area":
            effective_surface_area,
    }


# ================================================================
# STAGE 7 — ENERGY TRANSFER
# ================================================================

def stage_7_energy_transfer(
    electrical_stage: Dict[str, float],
    em_stage: Dict[str, Any],
    cryo_stage: Dict[str, float],
    mechanical_stage: Dict[str, float]
) -> Dict[str, float]:
    """
    Consolidates the physical state entering the chemical stage.

    IMPORTANT:
    The individual terms are kept separate rather than blindly
    added together. This makes it possible to identify which
    mechanism actually contributes to the predicted response.
    """

    electrical_power = (
        electrical_stage["electrical_power_w"]
    )

    current_density = (
        electrical_stage[
            "current_density_a_per_m2"
        ]
    )

    surface_multiplier = (
        mechanical_stage[
            "surface_area_multiplier"
        ]
    )

    fracture_activation = (
        mechanical_stage[
            "fracture_activation"
        ]
    )

    cryo_factor = (
        1.0
        + cryo_stage[
            "brittleness_factor"
        ]
    )

    # Effective transport metric.
    #
    # This is intentionally a composite experimental
    # parameter rather than a claim of fundamental physics.
    effective_activation_index = (
        electrical_power
        * surface_multiplier
        * cryo_factor
        * (1.0 + fracture_activation)
    )

    return {
        "electrical_power_w":
            electrical_power,

        "current_density_a_per_m2":
            current_density,

        "surface_area_multiplier":
            surface_multiplier,

        "fracture_activation":
            fracture_activation,

        "cryo_factor":
            cryo_factor,

        "effective_activation_index":
            effective_activation_index,
    }


# ================================================================
# STAGE 8 — MOLECULAR / CHEMICAL RESPONSE
# ================================================================

@dataclass
class ChemicalConfig:

    # Probability of chain fragmentation caused by mechanical
    # activation. This is phenomenological.
    base_fragmentation_probability: float = 0.30

    # Nominal activation-energy scale
    activation_energy_ev: float = 2.5

    # Fraction of activation energy potentially supplied
    # through the modeled electrical pathway.
    energy_coupling_efficiency: float = 0.01


def stage_8_molecular_response(
    material: MaterialConfig,
    energy_stage: Dict[str, float],
    chemical_cfg: ChemicalConfig
) -> Dict[str, Any]:

    rng = np.random.default_rng(42)

    # ------------------------------------------------------------
    # 1. Generate initial chain distribution
    # ------------------------------------------------------------

    initial_chains = rng.normal(
        material.initial_avg_chain_length,
        material.chain_length_std_dev,
        material.num_chains,
    )

    initial_chains = np.maximum(
        1,
        initial_chains
    ).astype(int)

    # ------------------------------------------------------------
    # 2. Mechanical chain fragmentation
    # ------------------------------------------------------------

    mechanical_factor = min(
        1.0,
        energy_stage[
            "surface_area_multiplier"
        ] / 2.0
    )

    fragmentation_probability = min(
        1.0,
        chemical_cfg.base_fragmentation_probability
        * mechanical_factor
    )

    fragmentation_mask = (
        rng.random(material.num_chains)
        < fragmentation_probability
    )

    liberated_chains = initial_chains.copy()

    liberated_chains[
        fragmentation_mask
    ] = np.maximum(
        1,
        (
            liberated_chains[
                fragmentation_mask
            ]
            * rng.uniform(
                0.3,
                0.7,
                fragmentation_mask.sum()
            )
        )
    ).astype(int)

    # ------------------------------------------------------------
    # 3. Estimate chemical activation
    # ------------------------------------------------------------

    nominal_bond_energy_j = (
        chemical_cfg.activation_energy_ev
        * EV_TO_J
    )

    effective_activation_index = (
        energy_stage[
            "effective_activation_index"
        ]
    )

    coupled_energy = (
        effective_activation_index
        * chemical_cfg.energy_coupling_efficiency
    )

    # This ratio is a normalized model quantity.
    # It is NOT a direct molecular bond-energy calculation.
    activation_ratio = (
        coupled_energy /
        max(
            nominal_bond_energy_j,
            1e-30
        )
    )

    # ------------------------------------------------------------
    # 4. Separate chain fragmentation from oxygen-bond cleavage
    # ------------------------------------------------------------

    predicted_oxygen_cleavage_probability = min(
        1.0,
        activation_ratio
    )

    oxygen_cleavage_mask = (
        rng.random(material.num_chains)
        < predicted_oxygen_cleavage_probability
    )

    return {

        "initial_chains":
            initial_chains,

        "liberated_chains":
            liberated_chains,

        "fragmentation_mask":
            fragmentation_mask,

        "fragmentation_probability":
            fragmentation_probability,

        "nominal_bond_energy_j":
            nominal_bond_energy_j,

        "coupled_energy":
            coupled_energy,

        "activation_ratio":
            activation_ratio,

        "predicted_oxygen_cleavage_probability":
            predicted_oxygen_cleavage_probability,

        "oxygen_cleavage_mask":
            oxygen_cleavage_mask,
    }


# ================================================================
# STAGE 9 — VALIDATION METRICS
# ================================================================

def stage_9_validation(
    results: Dict[str, Any]
) -> Dict[str, float]:

    chemical = results[
        "chemical"
    ]

    initial_count = len(
        chemical["initial_chains"]
    )

    fragmented_count = np.sum(
        chemical["fragmentation_mask"]
    )

    oxygen_cleaved_count = np.sum(
        chemical["oxygen_cleavage_mask"]
    )

    return {

        "fragmentation_fraction":
            fragmented_count /
            initial_count,

        "predicted_oxygen_cleavage_fraction":
            oxygen_cleaved_count /
            initial_count,

        "activation_ratio":
            chemical[
                "activation_ratio"
            ],
    }


# ================================================================
# MASTER STAGED PIPELINE
# ================================================================

@dataclass
class StagedSimulationConfig:

    material: MaterialConfig = field(
        default_factory=MaterialConfig
    )

    interface: InterfaceConfig = field(
        default_factory=InterfaceConfig
    )

    electrical: ElectricalConfig = field(
        default_factory=ElectricalConfig
    )

    cryogenic: CryogenicConfig = field(
        default_factory=CryogenicConfig
    )

    mechanical: MechanicalConfig = field(
        default_factory=MechanicalConfig
    )

    electromagnetic: EMConfig = field(
        default_factory=EMConfig
    )

    chemical: ChemicalConfig = field(
        default_factory=ChemicalConfig
    )


def run_staged_simulation(
    config: StagedSimulationConfig
) -> Dict[str, Any]:

    results = {}

    # ------------------------------------------------------------
    # STAGE 0
    # ------------------------------------------------------------

    results["baseline"] = {
        "initial_mass_kg":
            config.material.initial_mass_kg,

        "conductivity_s_per_m":
            config.material.conductivity_s_per_m,

        "resistivity_ohm_m":
            config.material.resistivity_ohm_m,
    }

    # ------------------------------------------------------------
    # STAGE 1 / 2
    # Geometry → Passive ΔV
    # ------------------------------------------------------------

    results["passive_potential"] = (
        stage_2_generate_passive_potential(
            config.interface
        )
    )

    # ------------------------------------------------------------
    # STAGE 3
    # Passive ΔV → Electrical transport
    # ------------------------------------------------------------

    results["electrical"] = (
        stage_3_electrical_transport(
            results[
                "passive_potential"
            ],
            config.electrical
        )
    )

    # ------------------------------------------------------------
    # STAGE 4
    # Electrical state → Cryogenic conditioning
    # ------------------------------------------------------------

    results["cryo"] = (
        stage_4_cryo_conditioning(
            config.material,
            config.cryogenic
        )
    )

    # ------------------------------------------------------------
    # STAGE 5
    # Cryogenic conditioning → Mechanical activation
    # ------------------------------------------------------------

    results["mechanical"] = (
        stage_5_mechanical_activation(
            results["cryo"],
            config.mechanical
        )
    )

    # ------------------------------------------------------------
    # STAGE 6
    # Mechanical activation → EM excitation
    # ------------------------------------------------------------

    results["electromagnetic"] = (
        stage_6_em_excitation(
            config.material,
            results["mechanical"],
            config.electromagnetic
        )
    )

    # ------------------------------------------------------------
    # STAGE 7
    # All prior states → Energy transfer
    # ------------------------------------------------------------

    results["energy_transfer"] = (
        stage_7_energy_transfer(
            results["electrical"],
            results["electromagnetic"],
            results["cryo"],
            results["mechanical"]
        )
    )

    # ------------------------------------------------------------
    # STAGE 8
    # Energy transfer → Molecular response
    # ------------------------------------------------------------

    results["chemical"] = (
        stage_8_molecular_response(
            config.material,
            results["energy_transfer"],
            config.chemical
        )
    )

    # ------------------------------------------------------------
    # STAGE 9
    # Chemical response → Validation metrics
    # ------------------------------------------------------------

    results["validation"] = (
        stage_9_validation(
            results
        )
    )

    return results


# ================================================================
# REPORTING
# ================================================================

def print_staged_report(
    results: Dict[str, Any]
):

    print("\n")
    print("=" * 70)
    print("STAGED MULTI-FIELD SIMULATION REPORT")
    print("=" * 70)

    print("\n[STAGE 2] PASSIVE POTENTIAL")
    print(
        f"Water potential: "
        f"{results['passive_potential']['water_potential_mv']:.4f} mV"
    )

    print(
        f"Oil potential: "
        f"{results['passive_potential']['oil_potential_mv']:.4f} mV"
    )

    print(
        f"Interface potential: "
        f"{results['passive_potential']['interface_potential_mv']:.4f} mV"
    )

    print(
        f"Passive ΔV: "
        f"{results['passive_potential']['passive_delta_v_mv']:.4f} mV"
    )

    print("\n[STAGE 3] ELECTRICAL TRANSPORT")

    print(
        f"Total ΔV: "
        f"{results['electrical']['total_delta_v_v']:.6e} V"
    )

    print(
        f"Current: "
        f"{results['electrical']['current_a']:.6e} A"
    )

    print(
        f"Power: "
        f"{results['electrical']['electrical_power_w']:.6e} W"
    )

    print("\n[STAGE 4] CRYOGENIC CONDITIONING")

    print(
        f"Thermal stress: "
        f"{results['cryo']['thermal_stress_mpa']:.3f} MPa"
    )

    print(
        f"Brittleness factor: "
        f"{results['cryo']['brittleness_factor']:.4f}"
    )

    print("\n[STAGE 5] MECHANICAL ACTIVATION")

    print(
        f"Fracture activation: "
        f"{results['mechanical']['fracture_activation']:.4f}"
    )

    print(
        f"Surface-area multiplier: "
        f"{results['mechanical']['surface_area_multiplier']:.4f}"
    )

    print("\n[STAGE 7] ENERGY TRANSFER")

    print(
        f"Effective activation index: "
        f"{results['energy_transfer']['effective_activation_index']:.6e}"
    )

    print("\n[STAGE 8] MOLECULAR RESPONSE")

    print(
        f"Chain fragmentation probability: "
        f"{results['chemical']['fragmentation_probability']:.4f}"
    )

    print(
        f"Predicted oxygen cleavage probability: "
        f"{results['chemical']['predicted_oxygen_cleavage_probability']:.6e}"
    )

    print("\n[STAGE 9] VALIDATION")

    print(
        f"Fragmentation fraction: "
        f"{results['validation']['fragmentation_fraction']:.4f}"
    )

    print(
        f"Predicted oxygen cleavage fraction: "
        f"{results['validation']['predicted_oxygen_cleavage_fraction']:.6e}"
    )

    print("=" * 70)


# ================================================================
# EXAMPLE EXECUTION
# ================================================================

if __name__ == "__main__":

    config = StagedSimulationConfig(

        electrical=ElectricalConfig(
            external_voltage_v=0.0,
            load_resistance_ohm=1000.0
        ),

        cryogenic=CryogenicConfig(
            target_temp_c=-196.0,
            freezing_rate_c_per_s=2.0
        ),

        electromagnetic=EMConfig(
            fundamental_freq_hz=50e3,
            max_harmonics=5,
            magnetic_field_tesla=5.0
        ),

        chemical=ChemicalConfig(
            activation_energy_ev=2.5,
            energy_coupling_efficiency=0.01
        )
    )

    results = run_staged_simulation(
        config
    )

    print_staged_report(
        results
    )

# Recommended Next Development Step

# The next revision should add a **mode-switching experiment framework** rather than immediately increasing model complexity.

# For example:

# ```python
# EXPERIMENTS = {

#     "baseline": {
#         "wicking": False,
#         "interface": False,
#         "external_voltage": False,
#         "cryo": False,
#         "mechanical": False,
#         "em": False,
#     },

#     "passive_potential": {
#         "wicking": True,
#         "interface": True,
#         "external_voltage": False,
#         "cryo": False,
#         "mechanical": False,
#         "em": False,
#     },

#     "electrical_only": {
#         "wicking": False,
#         "interface": False,
#         "external_voltage": True,
#         "cryo": False,
#         "mechanical": False,
#         "em": False,
#     },

#     "passive_plus_electrical": {
#         "wicking": True,
#         "interface": True,
#         "external_voltage": True,
#         "cryo": False,
#         "mechanical": False,
#         "em": False,
#     },

#     "passive_electrical_cryo": {
#         "wicking": True,
#         "interface": True,
#         "external_voltage": True,
#         "cryo": True,
#         "mechanical": False,
#         "em": False,
#     },

#     "full_system": {
#         "wicking": True,
#         "interface": True,
#         "external_voltage": True,
#         "cryo": True,
#         "mechanical": True,
#         "em": True,
#     },
# }
# ```

# The output should then compare:

# [
# \Delta V
# ]

# [
# I
# ]

# [
# P=IV
# ]

# [
# \text{surface-area increase}
# ]

# [
# \text{mechanical fragmentation}
# ]

# [
# \text{chemical activation index}
# ]

# [
# \text{measured oxygen-bond cleavage}
# ]

# This is the critical distinction I would make in the next version:

# [
# \boxed{\text{Model Prediction}\neq\text{Experimental Observation}}
# ]

# The simulation should predict a measurable outcome, while the experimental system determines whether that prediction is actually correct.
